# VeReMi One-Archive Semantic Decoder

This notebook decodes one original VeReMi `.tgz` archive before large-scale graph construction.

It investigates filename identifiers, A0/A1, ground-truth semantics, message matching, one-to-many reception, communication delay, and basic graph statistics.


In [ ]:
from pathlib import Path
import sys

# Locate repository root when notebook is executed from notebooks/
PROJECT_ROOT = Path.cwd()

if PROJECT_ROOT.name in {"notebooks", "analysis", "examples"}:
    PROJECT_ROOT = PROJECT_ROOT.parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))


In [ ]:
from pathlib import Path
import tarfile, json, re, math
import pandas as pd
import numpy as np

from src.config import RAW_DATA_ROOT, SEMANTIC_VALIDATION_ROOT
ROOT = RAW_DATA_ROOT
RESULTS_DIR = SEMANTIC_VALIDATION_ROOT
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

tgz_files = sorted(ROOT.glob("*.tgz"))
if not tgz_files:
    raise FileNotFoundError(f"No .tgz files found in {ROOT}")

archive_path = tgz_files[0]
print("Selected archive:", archive_path.name)
print("Results directory:", RESULTS_DIR)


In [ ]:
LOG_RE = re.compile(r"JSONlog-(?P<id1>\d+)-(?P<id2>\d+)-A(?P<aflag>\d+)\.json$")

def read_json_lines(tar, member):
    f = tar.extractfile(member)
    rows = []
    if f is None:
        return rows
    for raw_line in f:
        line = raw_line.decode("utf-8", errors="replace").strip()
        if not line:
            continue
        try:
            rows.append(json.loads(line))
        except json.JSONDecodeError:
            pass
    return rows

def euclidean_xy(a,b):
    if a is None or b is None or len(a)<2 or len(b)<2:
        return np.nan
    return ((float(a[0])-float(b[0]))**2 + (float(a[1])-float(b[1]))**2)**0.5


In [ ]:
with tarfile.open(archive_path, "r:gz") as tar:
    members=[m for m in tar.getmembers() if m.isfile()]
    log_members=[]
    ground_truth_member=None

    for m in members:
        short=Path(m.name).name
        if short=="GroundTruthJSONlog.json":
            ground_truth_member=m
            continue
        match=LOG_RE.search(short)
        if match:
            log_members.append({
                "member":m,
                "file_name":short,
                "id1":int(match.group("id1")),
                "id2":int(match.group("id2")),
                "A_flag":int(match.group("aflag"))
            })

log_index=pd.DataFrame([{
    "file_name":x["file_name"],"id1":x["id1"],"id2":x["id2"],
    "A_flag":x["A_flag"],"size_kb":round(x["member"].size/1024,3)
} for x in log_members])

print("Individual JSON logs:", len(log_index))
print("Ground truth present:", ground_truth_member is not None)
display(log_index.sort_values(["A_flag","id1"]))
log_index.to_csv(RESULTS_DIR/"json_log_index.csv", index=False)


In [ ]:
if ground_truth_member is None:
    raise ValueError("GroundTruthJSONlog.json not found")

with tarfile.open(archive_path,"r:gz") as tar:
    gt_rows=read_json_lines(tar, ground_truth_member)

gt=pd.DataFrame(gt_rows)
print("Ground-truth rows:", len(gt))
print("Columns:", gt.columns.tolist())
display(gt.head(10))

print("\nType distribution:")
display(gt["type"].value_counts(dropna=False).sort_index())

if "attackerType" in gt.columns:
    print("\nattackerType distribution:")
    display(gt["attackerType"].value_counts(dropna=False).sort_index())

gt.to_csv(RESULTS_DIR/"ground_truth_records.csv", index=False)


In [ ]:
file_summaries=[]
all_type3=[]
local_type2_by_file={}

with tarfile.open(archive_path,"r:gz") as tar:
    for info in log_members:
        rows=read_json_lines(tar, info["member"])
        if not rows:
            continue
        df=pd.DataFrame(rows)
        counts=df["type"].value_counts(dropna=False).to_dict()
        type2=df[df["type"]==2].copy()
        type3=df[df["type"]==3].copy()
        local_type2_by_file[info["file_name"]]=type2

        file_summaries.append({
            "file_name":info["file_name"],
            "id1":info["id1"],
            "id2":info["id2"],
            "A_flag":info["A_flag"],
            "total_records":len(df),
            "type2_local_records":int(counts.get(2,0)),
            "type3_received_records":int(counts.get(3,0)),
            "minimum_rcvTime":pd.to_numeric(df.get("rcvTime"),errors="coerce").min(),
            "maximum_rcvTime":pd.to_numeric(df.get("rcvTime"),errors="coerce").max()
        })

        if not type3.empty:
            type3["receiver_log_file"]=info["file_name"]
            type3["receiver_id1_candidate"]=info["id1"]
            type3["receiver_id2_candidate"]=info["id2"]
            type3["receiver_A_flag"]=info["A_flag"]
            all_type3.append(type3)

file_summary_df=pd.DataFrame(file_summaries)
display(file_summary_df.sort_values(["A_flag","id1"]))
file_summary_df.to_csv(RESULTS_DIR/"per_log_summary.csv", index=False)

received=pd.concat(all_type3, ignore_index=True)
print("Total received type-3 messages:", len(received))
print("Unique senders:", received["sender"].nunique())
print("Unique message IDs:", received["messageID"].nunique())


In [ ]:
# Infer whether id1 or id2 is the local/receiver vehicle identifier
gt_work=gt.copy()
gt_work["time"]=pd.to_numeric(gt_work["time"], errors="coerce")

receiver_tests=[]

for _,meta in log_index.iterrows():
    fname=meta["file_name"]
    local=local_type2_by_file.get(fname)
    if local is None or local.empty:
        continue

    local=local.copy()
    local["rcvTime"]=pd.to_numeric(local["rcvTime"],errors="coerce")
    local=local.dropna(subset=["rcvTime"])

    if len(local)>200:
        local=local.iloc[np.linspace(0,len(local)-1,200,dtype=int)]

    for candidate_name,candidate_id in [("id1",int(meta["id1"])),("id2",int(meta["id2"]))]:
        gt_candidate=gt_work[gt_work["sender"]==candidate_id].sort_values("time")

        if gt_candidate.empty:
            receiver_tests.append({
                "file_name":fname,"candidate":candidate_name,"candidate_id":candidate_id,
                "matched_points":0,"median_time_gap":np.nan,"median_position_error":np.nan
            })
            continue

        candidate_times=gt_candidate["time"].to_numpy()
        gaps=[]
        errs=[]

        for _,row in local.iterrows():
            t=float(row["rcvTime"])
            idx=np.searchsorted(candidate_times,t)
            options=[]
            if idx<len(candidate_times): options.append(idx)
            if idx>0: options.append(idx-1)
            if not options: continue

            best=min(options,key=lambda j:abs(candidate_times[j]-t))
            gtr=gt_candidate.iloc[best]
            gaps.append(abs(float(gtr["time"])-t))
            errs.append(euclidean_xy(row.get("pos"),gtr.get("pos")))

        receiver_tests.append({
            "file_name":fname,"candidate":candidate_name,"candidate_id":candidate_id,
            "matched_points":len(gaps),
            "median_time_gap":np.nanmedian(gaps) if gaps else np.nan,
            "median_position_error":np.nanmedian(errs) if errs else np.nan
        })

receiver_test_df=pd.DataFrame(receiver_tests)
display(receiver_test_df.head(40))
receiver_test_df.to_csv(RESULTS_DIR/"receiver_id_candidate_test.csv", index=False)

receiver_winners=(receiver_test_df.dropna(subset=["median_position_error"])
                  .sort_values(["file_name","median_position_error","median_time_gap"])
                  .groupby("file_name",as_index=False).first())

print("\nWinner counts:")
display(receiver_winners["candidate"].value_counts().rename_axis("winner").reset_index(name="file_count"))
display(receiver_winners.head(50))
receiver_winners.to_csv(RESULTS_DIR/"receiver_id_inference.csv", index=False)


In [ ]:
# Test ground-truth message uniqueness
sender_message_uniqueness=(gt.groupby(["sender","messageID"]).size()
                           .reset_index(name="frequency"))
duplicates=sender_message_uniqueness[sender_message_uniqueness["frequency"]>1]

print("Unique (sender,messageID) pairs:", len(sender_message_uniqueness))
print("Repeated pairs:", len(duplicates))
display(duplicates.head(30))
duplicates.to_csv(RESULTS_DIR/"duplicate_groundtruth_sender_message_pairs.csv", index=False)


In [ ]:
# Match type-3 received messages to type-4 ground truth
gt_cols=["sender","messageID","time","attackerType"]
for c in ["pos","pos_noise","spd","spd_noise"]:
    if c in gt.columns:
        gt_cols.append(c)

gt_small=gt[gt_cols].copy()

matched=received.merge(
    gt_small,
    on=["sender","messageID"],
    how="left",
    suffixes=("_received","_groundtruth"),
    indicator=True
)

matched["groundtruth_matched"]=matched["_merge"]=="both"
matched["send_gt_time_difference"]=(
    pd.to_numeric(matched["sendTime"],errors="coerce")-
    pd.to_numeric(matched["time"],errors="coerce")
).abs()

print("Received messages:",len(matched))
print("Matched:",int(matched["groundtruth_matched"].sum()))
print("Match percentage:",round(matched["groundtruth_matched"].mean()*100,4),"%")

display(matched.loc[matched["groundtruth_matched"],"send_gt_time_difference"].describe())

matched["communication_delay"]=(
    pd.to_numeric(matched["rcvTime"],errors="coerce")-
    pd.to_numeric(matched["sendTime"],errors="coerce")
)

print("\nCommunication delay:")
display(matched["communication_delay"].describe(percentiles=[.01,.05,.5,.95,.99]))

matched.to_parquet(RESULTS_DIR/"received_messages_matched_groundtruth.parquet", index=False)


In [ ]:
# Test one-to-many reception
reception_multiplicity=(matched.groupby(["sender","messageID"])
    .agg(receiver_log_count=("receiver_log_file","nunique"),
         received_copy_count=("receiver_log_file","size"))
    .reset_index())

print("Messages received by >1 receiver:",
      int((reception_multiplicity["receiver_log_count"]>1).sum()))
display(reception_multiplicity[["receiver_log_count","received_copy_count"]].describe())

reception_multiplicity.to_csv(
    RESULTS_DIR/"message_reception_multiplicity.csv", index=False
)


In [ ]:
# Audit A0/A1 without assuming semantics
a_flag_summary=(file_summary_df.groupby("A_flag")
    .agg(file_count=("file_name","count"),
         total_records=("total_records","sum"),
         total_type2=("type2_local_records","sum"),
         total_type3=("type3_received_records","sum"))
    .reset_index())

display(a_flag_summary)
a_flag_summary.to_csv(RESULTS_DIR/"A_flag_summary.csv", index=False)

receiver_map=receiver_winners[["file_name","candidate_id"]].rename(
    columns={"candidate_id":"inferred_receiver_id"}
)

file_receiver_audit=log_index.merge(receiver_map,on="file_name",how="left")
profiles=[]

for _,row in file_receiver_audit.iterrows():
    rid=row["inferred_receiver_id"]
    if pd.isna(rid):
        continue
    gtr=gt[gt["sender"]==int(rid)]
    vals=sorted(gtr["attackerType"].dropna().unique().tolist()) if not gtr.empty else []
    profiles.append({
        "file_name":row["file_name"],
        "A_flag":row["A_flag"],
        "inferred_receiver_id":int(rid),
        "groundtruth_attackerTypes_seen":str(vals),
        "groundtruth_records":len(gtr)
    })

profiles_df=pd.DataFrame(profiles)
display(profiles_df)
profiles_df.to_csv(RESULTS_DIR/"A_flag_vs_receiver_groundtruth.csv", index=False)


In [ ]:
# Build one-archive directed communication edge table
receiver_lookup=dict(zip(receiver_winners["file_name"],receiver_winners["candidate_id"]))

edge_df=matched.copy()
edge_df["receiver"]=edge_df["receiver_log_file"].map(receiver_lookup)
edge_df=edge_df.dropna(subset=["sender","receiver","sendTime","rcvTime"]).copy()

edge_df["sender"]=edge_df["sender"].astype(int)
edge_df["receiver"]=edge_df["receiver"].astype(int)
edge_df["edge_id"]=np.arange(len(edge_df))

edge_cols=[
    "edge_id","sender","receiver","messageID","sendTime","rcvTime",
    "communication_delay","RSSI","attackerType","receiver_log_file",
    "receiver_A_flag","groundtruth_matched"
]
edge_cols=[c for c in edge_cols if c in edge_df.columns]

edge_table=edge_df[edge_cols].copy()

vehicles=set(edge_table["sender"].unique()) | set(edge_table["receiver"].unique())

print("Communication edges:",len(edge_table))
print("Unique senders:",edge_table["sender"].nunique())
print("Unique receivers:",edge_table["receiver"].nunique())
print("Unique vehicles overall:",len(vehicles))
display(edge_table.head(20))

edge_table.to_parquet(RESULTS_DIR/"one_archive_communication_edges.parquet", index=False)


In [ ]:
graph_summary=pd.DataFrame({
    "metric":[
        "archive_name","json_log_files","groundtruth_records",
        "communication_edges","unique_vehicles","unique_senders",
        "unique_receivers","groundtruth_match_percentage",
        "A0_files","A1_files"
    ],
    "value":[
        archive_path.name,len(log_index),len(gt),len(edge_table),len(vehicles),
        edge_table["sender"].nunique(),edge_table["receiver"].nunique(),
        round(matched["groundtruth_matched"].mean()*100,6),
        int((log_index["A_flag"]==0).sum()),
        int((log_index["A_flag"]==1).sum())
    ]
})

display(graph_summary)
graph_summary.to_csv(RESULTS_DIR/"one_archive_graph_summary.csv", index=False)

print("Semantic decoding complete.")
print("Results saved to:", RESULTS_DIR)
